# ESCULTOR · Entrenar un detector de puntos para tus esculturas

Este cuaderno entrena un **detector de 14 puntos del cuerpo** (cráneo, barbilla, hombros, codos, muñecas, caderas, rodillas y tobillos) con los ejemplos que guardaste en la app (*Más ▸ 🧠 Guardar ejemplo para entrenar* y *⬇️ Exportar datos de entrenamiento*).

**Cómo usarlo (gratis, en Google Colab):**
1. Menú *Entorno de ejecución ▸ Cambiar tipo de entorno ▸ GPU (T4)*.
2. Ejecuta las celdas en orden (▶). En la celda 2 sube tu archivo `escultor-entrenamiento-AAAA-MM-DD.json`.
3. Al final se descarga `modelo-escultor.zip`. Mándamelo (o súbelo a la carpeta `modelo/` del repositorio) y lo conecto en la app.

**Cuántos ejemplos:** con menos de ~150 el resultado suele ser peor que el detector actual. Lo ideal: 150–300 fotos de **esculturas distintas**, en varias poses, luces y fondos, con los puntos **corregidos a mano** (modo ✋ Manual).


In [ ]:
# 1. Librerías
!pip -q install tensorflowjs==4.20.0
import json, base64, io, random, numpy as np, tensorflow as tf
from PIL import Image
print('TensorFlow', tf.__version__, '· GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 2. Sube el archivo exportado desde ESCULTOR
from google.colab import files
subido = files.upload()
datos = json.loads(list(subido.values())[0])
ej = datos['ejemplos']
print(len(ej), 'ejemplos')

In [ ]:
# 3. Preparar datos: imagen 256×256 y 14 puntos (coordenadas 0–1)
PUNTOS = ['top','chin','shL','shR','elL','elR','wrL','wrR','hipL','hipR','knL','knR','anL','anR']
TAM = 256
X, Y, PESO = [], [], []
for e in ej:
    c = e.get('cuerpo')
    if not c or not c.get('puntos'): continue
    pts = c['puntos']
    if any(pts.get(k) is None for k in PUNTOS): continue
    img = Image.open(io.BytesIO(base64.b64decode(e['img'].split(',')[1]))).convert('RGB')
    w, h = img.size; s = max(w, h)
    lienzo = Image.new('RGB', (s, s)); lienzo.paste(img, ((s - w)//2, (s - h)//2))
    X.append(np.asarray(lienzo.resize((TAM, TAM)), dtype=np.float32) / 255.)
    Y.append([[(pts[k]['x'] * w + (s - w)/2) / s, (pts[k]['y'] * h + (s - h)/2) / s] for k in PUNTOS])
    PESO.append(2.0 if 'manual' in c.get('origen', '') else 1.0)   # los corregidos a mano valen más
X, Y, PESO = np.array(X), np.array(Y, dtype=np.float32), np.array(PESO, dtype=np.float32)
print('Ejemplos útiles:', len(X), '· corregidos a mano:', int((PESO > 1).sum()))
assert len(X) >= 20, 'Hacen falta más ejemplos (mínimo 20 para probar; 150+ para que valga la pena).'

In [ ]:
# 4. Mapas de calor (una gaussiana por punto) y aumento de datos
HM = 64
def mapas(y):
    g = np.zeros((HM, HM, len(PUNTOS)), np.float32); yy, xx = np.mgrid[0:HM, 0:HM]
    for i, (px, py) in enumerate(y): g[..., i] = np.exp(-((xx - px*HM)**2 + (yy - py*HM)**2) / (2 * 1.5**2))
    return g
ESPEJO = [0,1,3,2,5,4,7,6,9,8,11,10,13,12]
def aumentar(x, y):
    if random.random() < 0.5: x = x[:, ::-1]; y = y[ESPEJO].copy(); y[:, 0] = 1 - y[:, 0]
    x = np.clip(x * random.uniform(0.7, 1.3) + random.uniform(-0.1, 0.1), 0, 1)
    return x, y
idx = np.random.permutation(len(X)); nval = max(2, len(X)//8)
val, tr = idx[:nval], idx[nval:]
def lotes(ids, aum):
    while True:
        np.random.shuffle(ids)
        for i in range(0, len(ids) - 7, 8):
            xs, hs, ws = [], [], []
            for j in ids[i:i+8]:
                x, y = (aumentar(X[j], Y[j]) if aum else (X[j], Y[j])); xs.append(x); hs.append(mapas(y)); ws.append(PESO[j])
            yield np.array(xs), np.array(hs), np.array(ws)

In [ ]:
# 5. Modelo: MobileNetV2 (preentrenado) + cabeza de mapas de calor
base = tf.keras.applications.MobileNetV2(input_shape=(TAM, TAM, 3), include_top=False, weights='imagenet', alpha=0.75)
x = base.get_layer('block_13_expand_relu').output            # 16×16
for f in (128, 64):
    x = tf.keras.layers.Conv2DTranspose(f, 4, strides=2, padding='same', activation='relu')(x)   # 32 → 64
salida = tf.keras.layers.Conv2D(len(PUNTOS), 1, activation='sigmoid')(x)
modelo = tf.keras.Model(base.input, salida)
modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
modelo.summary(line_length=100)

In [ ]:
# 6. Entrenar
pasos = max(10, len(tr)//8)
hist = modelo.fit(lotes(tr.copy(), True), steps_per_epoch=pasos, epochs=60,
                  validation_data=lotes(val.copy(), False), validation_steps=max(1, len(val)//8),
                  callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)], verbose=2)

In [ ]:
# 7. ¿Qué tan bien acierta? Error medio en cabezas (1 cabeza = cráneo→barbilla)
def predecir(x):
    h = modelo.predict(x[None], verbose=0)[0]
    return np.array([[np.argmax(h[..., i]) % HM / HM, np.argmax(h[..., i]) // HM / HM] for i in range(len(PUNTOS))])
err = []
for j in val:
    p, y = predecir(X[j]), Y[j]; cab = np.linalg.norm(y[0] - y[1]) or 1e-6
    err.append(np.mean(np.linalg.norm(p - y, axis=1)) / cab)
print(f'Error medio: {np.mean(err):.2f} cabezas (bueno si < 0,15)')

In [ ]:
# 8. Exportar para la app (TensorFlow.js) y descargar
import tensorflowjs as tfjs, shutil
tfjs.converters.save_keras_model(modelo, 'modelo-escultor')
json.dump({'puntos': PUNTOS, 'entrada': TAM, 'mapa': HM, 'ejemplos': int(len(X))}, open('modelo-escultor/info.json', 'w'))
shutil.make_archive('modelo-escultor', 'zip', 'modelo-escultor')
files.download('modelo-escultor.zip')